# Working With datasets

This tutorial is a sequel to [Tutorial 01](https://lc.llnl.gov/jupyter/user/cdoutrix/notebooks/git/Kosh/examples/Example_01_Add_Data_To_Datasets.ipynb) which should have been successfully ran before this tutotrial.

In this tutorial we will open a store, look for some datasets of interest, search for failed nodes and time, and mark the datasets as failed if necessary.


## Connect to store (using sina local file)


In [1]:
from  kosh import KoshStore
import os

# local tutorial sql file
kosh_example_sql_file = "kosh_example.sql"

# connect to store
store = KoshStore(engine="sina", username=os.environ["USER"], sql='sql', db_path=kosh_example_sql_file)

['9e5eb5ef19a5462f9cf5b9549758d8b6']


## Looping through datasets

Let's look for our "IBM project"-related datsets

In [2]:
datasets = store.search(project="IBM")
print("We identified {} possible datasets".format(len(datasets)))

We identified 320 possible datasets


## Working with datasets and files.

Now we are going to identify failed nodes and their failure cycles.


In [3]:
import numpy
import h5py
import json
import aml_dmt
from tqdm import tqdm
import glob


In [4]:
json_file = "/p/lscratchh/cdoutrix/cdoutrix/IBM/workaround/bad_nodes.json"

pbar = tqdm(datasets)
for ds in pbar:
    pbar.set_description("looking at: {:45}".format(ds.name))
    hdf5 = ds.search(mime_type="hdf5")
    if len(hdf5)>0:
        h5 = hdf5[0]
        ds.bad_nodes = aml_dmt.ibm.identify_bad_nodes(h5.uri, jsonfile=json_file, jsonkey=ds.name, verbose=False)
        h5file = h5py.File(h5.uri, "r")
        ds.cycles = h5file["node"]["metrics_0"].shape[0]
        ds.nodes = h5file["node"]["metrics_0"].shape[1]
        # look up end time in slurm output file
        slurm = glob.glob("/p/lscratchh/cdoutrix/cdoutrix/IBM/workaround/base/{}/slurm*out",format(ds.name))
        last_time = None
        last_cycle = None
        if len(slurm)>0:
            slurm = slurm[0]
            with open(slurm) as fslurm:
                for ln in f.readlines():
                    if "Print cycle =" in ln:
                        sp = ln.split()
                        last_cycle = int(sp[3])
                        last_time = float(sp[6])
        ds.last_time = last_time
        ds.last_cycle = last_cycle
        ds.goal_time = 650000.  # Fixed in our experiments
        #ds.node_metrics = [aml_dmt.ibm.de_anonymize(k, h5.uri.replace(".hdf5",".json")) for k in h5file["node"]]


looking at: molar.0.7_shock.1.3_taper.0.8_skew.0.4       : 100%|██████████| 320/320 [03:49<00:00,  1.40it/s]


In [ ]:
ds.bad_nodes, ds.cycles

In [6]:
error = '/p/lscratchh/cdoutrix/cdoutrix/IBM/workaround/base/molar.1_shock.1.7_taper.0.6_skew.0.8/slurm-340974.out'
no_error = '/p/lscratchh/cdoutrix/cdoutrix/IBM/workaround/base/molar.1_shock.1.3_taper.0.6_skew.0.6/slurm-340933.out'
error_goal_reached = '/p/lscratchh/cdoutrix/cdoutrix/IBM/workaround/base/molar.0.3_shock.1.3_taper.0.8_skew.0.9/slurm-340778.out'

In [7]:
#h5 = store.search(name=error_goal_reached.split("/")[-2])[0]
#h5 = h5.search(type="hdf5")[0].uri
#print("Using:", h5)
#bad = identify_bad_nodes(h5)

In [8]:
bad_nodes['molar.0.0_shock.1.3_taper.0.8_skew.0.6'].result()

NameError: name 'bad_nodes' is not defined